# Limpieza de Datos: AVIAN-MAMM
### Minería de Datos · Vigilancia de influenza aviar en mamíferos

Este notebook toma el CSV original (`datos/latest-outbreaks-mammals.csv`) y deja un CSV limpio (`datos/avian_clean.csv`).
**El archivo original no se edita a mano:** toda corrección ocurre aquí, para que el proceso se pueda repetir.

## 1. Importar librerías

In [5]:
import numpy as np
import pandas as pd

## 2. Carga del dataset
AVIAN-MAMM es una base abierta (CC BY-SA 4.0) construida a partir de los reportes de WAHIS (OMSA).
Cada fila es un evento de influenza aviar reportado en mamíferos.

In [6]:
# Busca datos/ desde la carpeta del proyecto o desde codigo/.
from pathlib import Path

dataset_candidates = [
    Path("datos/latest-outbreaks-mammals.csv"),
    Path("../datos/latest-outbreaks-mammals.csv"),
]
dataset_path = next((path for path in dataset_candidates if path.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError(
        f"No se encontró latest-outbreaks-mammals.csv desde {Path.cwd()}"
    )

data_dir = dataset_path.parent
df = pd.read_csv(dataset_path)

print(df.shape)
df.head()

(379, 48)


,url_event,url_pdf,report_number,report_id,report_date,event_id,genotype_serotype_subtype,test_name_1,laboratory_name_1,first_result_date_1,...,total_cases,total_deaths,total_killed_and_disposed_of,total_slaughtered_killed_for_commercial_use,total_vaccinated,epidemiological_unit,detailed_characterisation,affected_population_description,control_measures,outcome
0,https://wahis.woah.org/#/in-event/4185/dashboard,https://github.com/fbranda/avian-ani/blob/main...,IN,IN_153005,2021-12-17,4185,H5N1,Real-time polymerase chain reaction (real-time...,Estonian Veterinary and Food Laboratory (VFL),44525,...,1.0,1.0,NaN,NaN,NaN,Not applicable,NaN,NaN,No control measure applied,death
1,https://wahis.woah.org/#/in-event/5037/dashboard,https://github.com/fbranda/avian-mamm/blob/mai...,FUR_16,FUR_162547,2023-08-30,5037,H5N1,Real-time reverse transcription polymerase cha...,National Reference and OIE/FAO Laboratory for ...,45028,...,1.0,1.0,NaN,NaN,NaN,Natural park,NaN,Animal with symptoms died shortly after discov...,"Screening, Control of wildlife reservoirs",death
2,https://wahis.woah.org/#/in-event/5037/dashboard,https://github.com/fbranda/avian-mamm/blob/mai...,FUR_16,FUR_162547,2023-08-30,5037,H5N1,Real-time reverse transcription polymerase cha...,National Reference and OIE/FAO Laboratory for ...,45028,...,1.0,1.0,NaN,NaN,NaN,Natural park,NaN,HPAI H5N1 was detected in a Red Fox (vulpes vu...,"Screening, Control of wildlife reservoirs",death
3,https://wahis.woah.org/#/in-event/5037/dashboard,https://github.com/fbranda/avian-mamm/blob/mai...,FUR_16,FUR_162547,2023-08-30,5037,H5N1,Virus neutralisation tests (Ab VNT),National Reference and OIE/FAO Laboratory for ...,45028,...,5.0,NaN,NaN,NaN,NaN,Farm,NaN,The highly pathogenic avian influenza virus (H...,"Quarantine, Surveillance within the restricted...",positive
4,https://wahis.woah.org/#/in-event/5037/dashboard,https://github.com/fbranda/avian-mamm/blob/mai...,FUR_16,FUR_162547,2023-08-30,5037,H5N1,Virus neutralisation tests (Ab VNT),National Reference and OIE/FAO Laboratory for ...,45028,...,1.0,NaN,NaN,NaN,NaN,Farm,NaN,The highly pathogenic avian influenza virus (H...,"Quarantine, Surveillance within the restricted...",positive


## 3. Inspección inicial
Revisamos tipos de dato y valores faltantes antes de tocar nada.

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 379 entries, 0 to 378
Data columns (total 48 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   url_event                                    379 non-null    str    
 1   url_pdf                                      379 non-null    str    
 2   report_number                                374 non-null    str    
 3   report_id                                    379 non-null    str    
 4   report_date                                  379 non-null    str    
 5   event_id                                     379 non-null    int64  
 6   genotype_serotype_subtype                    379 non-null    str    
 7   test_name_1                                  353 non-null    str    
 8   laboratory_name_1                            353 non-null    str    
 9   first_result_date_1                          349 non-null    str    
 10  latest_result

In [8]:
# Columnas con más valores faltantes (en %)
(df.isnull().mean() * 100).round(1).sort_values(ascending=False).head(15)

detailed_characterisation                      92.1
latest_result_date_3                           91.3
test_name_3                                    91.0
first_result_date_3                            91.0
laboratory_name_3                              91.0
result_3                                       91.0
total_susceptible                              89.4
total_vaccinated                               83.4
total_slaughtered_killed_for_commercial_use    83.1
latest_result_date_2                           70.4
first_result_date_2                            69.7
laboratory_name_2                              69.7
test_name_2                                    69.7
total_killed_and_disposed_of                   67.3
result_2                                       66.2
dtype: float64

Observaciones: la columna `longitude` aparece como texto (`str`) y varias columnas tienen más de 80 % de valores vacíos.
Además, `outcome` es la variable que queremos estudiar (muerte o positivo), así que hay que cuidar que no se filtre a los predictores.

## 4. Limpieza de datos

### 4.1 Conversión de tipos: longitud
`longitude` es texto porque 10 valores usan coma decimal y un espacio especial (por ejemplo `' -79,5'`).
La convertimos a número.

In [9]:
print("Tipo antes:", df['longitude'].dtype)

con_coma = df['longitude'].astype(str).str.contains(',').sum()
print(f"Valores con coma decimal: {con_coma}")

df['longitude'] = pd.to_numeric(
    df['longitude'].astype(str).str.replace('\xa0', '', regex=False).str.replace(',', '.', regex=False).str.strip(),
    errors='coerce')

print("Tipo después:", df['longitude'].dtype)
print("Longitudes vacías tras convertir:", df['longitude'].isnull().sum())

Tipo antes: str
Valores con coma decimal: 10
Tipo después: float64
Longitudes vacías tras convertir: 0


### 4.2 Valores fuera de rango: latitud
Una latitud válida está entre -90 y 90. Buscamos las que no cumplen.

In [10]:
fuera_rango = df['latitude'].abs() > 90
df.loc[fuera_rango, ['country_name', 'location', 'latitude', 'longitude']]

,country_name,location,latitude,longitude
122,United States of America,Grant County,42841.0,-90.709


Es un evento de Grant County (EE.UU.) con latitud `42841`. Se perdió el punto decimal: debería ser `42,841`,
que coincide con ese condado y con su longitud. Dividimos por 1000.

In [11]:
df.loc[fuera_rango, 'latitude'] = df.loc[fuera_rango, 'latitude'] / 1000

# Comprobación: todas las coordenadas deben estar en rango
assert df['latitude'].between(-90, 90).all()
assert df['longitude'].between(-180, 180).all()
df.loc[fuera_rango, ['location', 'latitude', 'longitude']]

,location,latitude,longitude
122,Grant County,42.841,-90.709


### 4.3 Fechas
Convertimos las fechas a tipo `datetime` y creamos el mes de inicio y la duración del brote.

In [12]:
for columna in ['report_date', 'start_date_outbreak', 'end_date_outbreak']:
    df[columna] = pd.to_datetime(df[columna], errors='coerce')

df['start_month'] = df['start_date_outbreak'].dt.to_period('M').astype(str)
df['duration_days'] = (df['end_date_outbreak'] - df['start_date_outbreak']).dt.days

print(f"Eventos sin fecha de cierre: {df['end_date_outbreak'].isnull().mean():.0%}")
df[['start_date_outbreak', 'end_date_outbreak', 'start_month', 'duration_days']].head()

Eventos sin fecha de cierre: 30%


,start_date_outbreak,end_date_outbreak,start_month,duration_days
0,2021-11-08,2021-11-25,2021-11,17.0
1,2023-04-03,NaT,2023-04,NaN
2,2023-04-07,NaT,2023-04,NaN
3,2023-04-18,NaT,2023-04,NaN
4,2023-04-18,NaT,2023-04,NaN


La duración solo se puede calcular donde hay fecha de cierre (alrededor del 70 % de los eventos).

### 4.4 Eliminación de duplicados

In [13]:
print(f"Cantidad de filas antes del drop: {len(df)}")
df = df.drop_duplicates()
print(f"Cantidad de filas después del drop: {len(df)}")

# label_id se repite, pero NO son duplicados: son especies o ubicaciones distintas dentro del mismo brote
print("label_id repetidos:", df['label_id'].duplicated().sum())

Cantidad de filas antes del drop: 379
Cantidad de filas después del drop: 379
label_id repetidos: 30


### 4.5 Unificar nombres de especies
Hay nombres distintos para la misma especie (`Cats` y `Domestic cat`, por ejemplo).

In [14]:
print("Especies distintas antes:", df['species'].nunique())
df['species'].value_counts().head(12)

Especies distintas antes: 37


species
Red Fox                             141
Striped Skunk                        40
South-American sea lion              37
Harbor Seal                          21
Arctic Fox                           20
Puma                                 17
Cats                                 13
Racoon (Northern raccoon)            13
Bovine                               13
Bobcat                                7
American Mink                         5
American Black Bear (black bear)      5
Name: count, dtype: int64

In [15]:
EQUIVALENCIAS = {
    "Cats": "Cat", "Domestic cat": "Cat", "Dogs": "Dog",
    "Racoon (Northern raccoon)": "Raccoon", "Bovine": "Cattle",
    "American Black Bear (black bear)": "Black bear", "Brown bear (Grizzly Bear)": "Brown bear",
    "North American river otter": "River otter", "American marten": "Marten",
    "European Pine Marten": "Marten", "South-American sea lion": "South American sea lion",
    "South American Coati": "Coati", "Huillin": "Huillin (southern river otter)",
}
df['species_clean'] = df['species'].str.strip().replace(EQUIVALENCIAS)
print("Especies distintas después:", df['species_clean'].nunique())

Especies distintas después: 35


Agrupamos las especies en grupos biológicos (criterio nuestro, se puede ajustar). Esto reduce las categorías para el análisis.
Además, `Brown skua` y `Kelp Gull` son **aves** (error de la fuente en una base de mamíferos): las marcamos en vez de borrarlas.

In [16]:
GRUPOS = {
    "Red Fox": "Canid", "Arctic Fox": "Canid", "Coyote": "Canid", "Raccoon dog": "Canid", "Dog": "Canid",
    "Striped Skunk": "Mustelid/skunk", "American Mink": "Mustelid/skunk", "Fisher": "Mustelid/skunk",
    "European Polecat": "Mustelid/skunk", "Marten": "Mustelid/skunk", "Mustelidae": "Mustelid/skunk",
    "River otter": "Mustelid/skunk", "Marine otter": "Mustelid/skunk", "Huillin (southern river otter)": "Mustelid/skunk",
    "South American sea lion": "Marine mammal", "Harbor Seal": "Marine mammal", "Gray Seal": "Marine mammal",
    "Northern Fur Seal": "Marine mammal", "Bottlenose dolphin": "Marine mammal",
    "Puma": "Felid", "Bobcat": "Felid", "Cat": "Felid", "Tiger": "Felid", "Lion": "Felid", "Amur Leopard": "Felid",
    "Black bear": "Ursid", "Brown bear": "Ursid", "Polar bear": "Ursid",
}
df['species_group'] = df['species_clean'].map(GRUPOS).fillna("Other")
df['is_bird'] = df['species_clean'].isin(["Brown skua", "Kelp Gull"])

df['species_group'].value_counts()

species_group
Canid             168
Marine mammal      63
Mustelid/skunk     57
Felid              44
Other              37
Ursid              10
Name: count, dtype: int64

### 4.6 Valores faltantes en variables categóricas
En `epidemiological_unit` hay 2 valores `-` y en `typology_of_animal` hay 1 vacío. No los adivinamos: los marcamos como `Unknown`.

In [17]:
print(df['epidemiological_unit'].value_counts())
print()
print(df['typology_of_animal'].value_counts(dropna=False))

epidemiological_unit
Not applicable    217
Other              91
Farm               51
Zoo                 6
Body of water       6
Natural park        3
Backyard            3
-                   2
Name: count, dtype: int64

typology_of_animal
Wild        352
Domestic     26
NaN           1
Name: count, dtype: int64


In [18]:
df['epidemiological_unit'] = df['epidemiological_unit'].replace({'-': np.nan}).fillna('Unknown')
df['typology_of_animal'] = df['typology_of_animal'].fillna('Unknown')

print(df['epidemiological_unit'].value_counts())

epidemiological_unit
Not applicable    217
Other              91
Farm               51
Zoo                 6
Body of water       6
Natural park        3
Backyard            3
Unknown             2
Name: count, dtype: int64


### 4.7 Eventos de Finlandia en granjas
33 eventos de Finlandia figuran como `Wild` (silvestres), pero todos ocurren en granjas (`Farm`) y las especies son
zorros, visones y perros mapache. Los marcamos para poder separarlos de los silvestres reales.

In [19]:
finlandia = df[df['country_name'] == 'Finland']
finlandia.groupby(['species', 'typology_of_animal', 'epidemiological_unit']).size()

species        typology_of_animal  epidemiological_unit
American Mink  Wild                Farm                     3
Arctic Fox     Wild                Farm                    20
Raccoon dog    Wild                Farm                     3
Red Fox        Wild                Farm                     7
dtype: int64

In [20]:
en_granja = (df['country_name'] == 'Finland') & (df['epidemiological_unit'] == 'Farm')

df['is_farmed_wild_species'] = en_granja
df['typology_adj'] = np.where(en_granja, 'Farmed', df['typology_of_animal'])

print(f"Eventos marcados: {en_granja.sum()}")
df['typology_adj'].value_counts()

Eventos marcados: 33


typology_adj
Wild        319
Farmed       33
Domestic     26
Unknown       1
Name: count, dtype: int64

## 5. Codificación de variables categóricas: medidas de control
`control_measures` es texto libre con combinaciones de frases (por ejemplo `Disinfection, Official disposal ...`).
Buscamos cada medida en el texto y creamos una columna 0/1 por medida.

In [21]:
df['control_measures'].value_counts().head(5)

control_measures
Disinfection, Official disposal of carcasses, by-products and waste, Surveillance within the restricted zone                                                                                                                                                                                                                                                   159
No control measure applied                                                                                                                                                                                                                                                                                                                                      59
Surveillance outside the restricted zone, Control of wildlife reservoirs, Official disposal of carcasses, by-products and waste, Disinfection, Surveillance within the restricted zone, Disinfection, Surveillance within the restricted zone, Official disposal of carcasses, by

In [22]:
MEDIDAS = {
    'ctl_disinfection': 'Disinfection',
    'ctl_disposal': 'Official disposal',
    'ctl_surv_within': 'Surveillance within',
    'ctl_surv_outside': 'Surveillance outside',
    'ctl_wildlife_ctrl': 'Control of wildlife',
    'ctl_selective_kill': 'Selective killing',
    'ctl_quarantine': 'Quarantine',
    'ctl_stamping_out': 'Stamping out',
    'ctl_zoning': 'Zoning',
    'ctl_screening': 'Screening',
}
texto = df['control_measures'].fillna('')

for columna, patron in MEDIDAS.items():
    df[columna] = texto.str.contains(patron, case=False, regex=True).astype(int)

df['ctl_none'] = texto.str.contains('No control measure', case=False).astype(int)
df['ctl_unknown'] = df['control_measures'].isnull().astype(int)

# Cantidad de medidas aplicadas (vacío si no hay información)
df['ctl_count'] = df[list(MEDIDAS)].sum(axis=1)
df.loc[df['ctl_unknown'] == 1, 'ctl_count'] = np.nan

print(f"Eventos sin información de medidas: {df['ctl_unknown'].sum()} (no se asumen 'sin medidas')")
df[list(MEDIDAS) + ['ctl_none', 'ctl_unknown', 'ctl_count']].head()

Eventos sin información de medidas: 68 (no se asumen 'sin medidas')


,ctl_disinfection,ctl_disposal,ctl_surv_within,ctl_surv_outside,ctl_wildlife_ctrl,ctl_selective_kill,ctl_quarantine,ctl_stamping_out,ctl_zoning,ctl_screening,ctl_none,ctl_unknown,ctl_count
0,0,0,0,0,0,0,0,0,0,0,1,0,0.0
1,0,0,0,0,1,0,0,0,0,1,0,0,2.0
2,0,0,0,0,1,0,0,0,0,1,0,0,2.0
3,1,1,1,0,0,0,1,1,1,0,0,0,6.0
4,1,1,1,0,0,0,1,1,1,0,0,0,6.0


## 6. Etiqueta: ¿el evento fue letal?
`outcome` puede ser `death`, `positive` o `not certified`. Creamos `lethal`:
1 = muerte, 0 = positivo, y vacío para los `not certified` (se analizan aparte).

In [23]:
df['outcome'] = df['outcome'].str.strip().str.lower()
df['lethal'] = df['outcome'].map({'death': 1, 'positive': 0})
df['certified'] = df['outcome'].ne('not certified')

df['outcome'].value_counts()

outcome
death            329
positive          34
not certified     16
Name: count, dtype: int64

**Ojo con la fuga de información:** según el diccionario de datos, `outcome` se calcula a partir de los conteos de casos y muertes.
Por eso `total_deaths` y `total_killed_and_disposed_of` **no deben usarse como predictores** de `lethal`.

## 7. Valores faltantes numéricos
`total_cases` y `total_deaths` tienen valores faltantes. En clases imputamos con `SimpleImputer`; aquí **no imputamos todavía**:
si lo hiciéramos antes de dividir en entrenamiento y prueba, el modelo vería información del conjunto de prueba.
La imputación se hace después, dentro de cada partición. Dejamos banderas que marcan cuáles faltaban y una versión logarítmica de los casos
(los casos son muy asimétricos, como se ve con `describe()`).

In [24]:
df[['total_cases', 'total_deaths']].describe()

,total_cases,total_deaths
count,358.000000,287.000000
mean,1.754190,1.891986
std,1.927516,2.759378
min,0.000000,0.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,17.000000,27.000000


In [25]:
for columna in ['total_cases', 'total_deaths']:
    df[f'{columna}_missing'] = df[columna].isnull().astype(int)

df['log_cases'] = np.log1p(df['total_cases'])

df[['total_cases', 'total_cases_missing', 'log_cases', 'total_deaths_missing']].head()

,total_cases,total_cases_missing,log_cases,total_deaths_missing
0,1.0,0,0.693147,0
1,1.0,0,0.693147,0
2,1.0,0,0.693147,0
3,5.0,0,1.791759,1
4,1.0,0,0.693147,1


## 8. Selección de columnas
Descartamos:
- columnas con más de 80 % de valores vacíos (casi no tienen información);
- URLs, identificadores y texto largo (siguen disponibles en el CSV original).

In [26]:
muchos_nulos = df.columns[df.isnull().mean() > 0.80].tolist()
print("Más de 80 % vacías:", muchos_nulos)

ids_y_texto = ['url_event', 'url_pdf', 'report_number', 'country_iso3_digit', 'affected_population_description']

df_limpio = df.drop(columns=muchos_nulos + ids_y_texto)
print(df.shape, '->', df_limpio.shape)

Más de 80 % vacías: ['test_name_3', 'laboratory_name_3', 'first_result_date_3', 'latest_result_date_3', 'result_3', 'total_susceptible', 'total_slaughtered_killed_for_commercial_use', 'total_vaccinated', 'detailed_characterisation']
(379, 73) -> (379, 59)


## 9. Dataset limpio

In [27]:
df_limpio.info()

<class 'pandas.DataFrame'>
RangeIndex: 379 entries, 0 to 378
Data columns (total 59 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   report_id                       379 non-null    str           
 1   report_date                     379 non-null    datetime64[us]
 2   event_id                        379 non-null    int64         
 3   genotype_serotype_subtype       379 non-null    str           
 4   test_name_1                     353 non-null    str           
 5   laboratory_name_1               353 non-null    str           
 6   first_result_date_1             349 non-null    str           
 7   latest_result_date_1            352 non-null    str           
 8   result_1                        352 non-null    str           
 9   test_name_2                     115 non-null    str           
 10  laboratory_name_2               115 non-null    str           
 11  first_result_date

In [28]:
print("lethal:")
print(df_limpio['lethal'].value_counts(dropna=False))
print()
print("Valores faltantes que quedan (solo columnas con alguno):")
faltantes = df_limpio.isnull().sum()
faltantes[faltantes > 0].sort_values(ascending=False)

lethal:
lethal
1.0    329
0.0     34
NaN     16
Name: count, dtype: int64

Valores faltantes que quedan (solo columnas con alguno):


latest_result_date_2              267
test_name_2                       264
laboratory_name_2                 264
first_result_date_2               264
total_killed_and_disposed_of      255
result_2                          251
outbreak_reference                213
third_administrative_division     210
duration_days                     112
end_date_outbreak                 112
total_deaths                       92
ctl_count                          68
control_measures                   68
first_result_date_1                30
result_1                           27
latest_result_date_1               27
test_name_1                        26
laboratory_name_1                  26
log_cases                          21
total_cases                        21
lethal                             16
second_administrative_division      8
dtype: int64

### Guardar

In [29]:
df_limpio.to_csv(data_dir / "avian_clean.csv", index=False)
print("Guardado: datos/avian_clean.csv", df_limpio.shape)

Guardado: datos/avian_clean.csv (379, 59)


## Resumen de lo que hicimos
| Problema | Solución |
|---|---|
| Longitud como texto con coma decimal (10 casos) | Se quitó el espacio especial, coma → punto, `to_numeric` |
| Latitud `42841` (1 caso) | Se dividió por 1000 y se validó el rango (-90, 90) |
| Especies con nombres duplicados | Diccionario de equivalencias (37 → 35) y grupo de especie |
| Aves en una base de mamíferos | Marcadas con `is_bird` |
| `-` y vacíos en categóricas | Marcados como `Unknown` |
| Finlandia en granjas registrada como silvestre | Bandera `is_farmed_wild_species` y `typology_adj` |
| Medidas de control en texto libre | 10 columnas 0/1 + `ctl_none`, `ctl_unknown`, `ctl_count` |
| Etiqueta | `lethal` (1 / 0 / vacío para `not certified`) |
| Faltantes numéricos | No se imputan aquí; banderas `*_missing` y `log_cases` |
| Columnas con más de 80 % vacías, URLs e ids | Descartadas |